# RAG passo a passo: Educação Financeira para Mulheres

Este notebook executa o mesmo pipeline dos scripts, **uma etapa por célula**, mostrando o que acontece por dentro. Use-o para estudar; para gerar os entregáveis, use os scripts (`ingerir.py`, `consultar.py`, `avaliar.py`).

**Antes de rodar:** `pip install -r requirements.txt`, Ollama aberto, `ollama pull bge-m3` e `ollama pull llama3.1:8b`, `python baixar_fontes.py` e `python ingerir.py` executados pelo menos uma vez.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

from rag import config
from rag.catalogo import carregar_catalogo
print(json.dumps(config.resumo(), indent=2, ensure_ascii=False))

## Etapa 0: A base curada

O catálogo define as fontes. Cada uma tem metadados que vão acompanhar todos os seus chunks.

In [ ]:
fontes = carregar_catalogo()
print(f"{len(fontes)} fontes no catálogo; {sum(f.foco_mulheres for f in fontes)} com foco em mulheres\n")
for f in fontes:
    status = "✓" if f.caminho_bruto.exists() else "✗"
    print(f"{status} {f.id:42} {f.instituicao[:35]:35} {f.ano}")

## Etapa 1: Carregar

Veja o texto **bruto** que sai do PDF, com hífens, quebras de linha e cabeçalhos repetidos.

In [ ]:
from rag.carregadores import carregar

fonte = next(f for f in fontes if f.id == "bcb_caderno_educacao_financeira")
unidades_brutas = carregar(fonte.caminho_bruto, fonte.formato)
print(f"{len(unidades_brutas)} páginas extraídas\n")
PAGINA = 10   # troque para ver outras páginas
print(repr(unidades_brutas[PAGINA].texto[:800]))

## Etapa 2: Limpar

Compare a mesma página antes e depois. Repare nas linhas detectadas como cabeçalho/rodapé.

In [ ]:
from rag.limpeza import limpar_unidades, linhas_repetidas

print("Linhas repetidas detectadas:", linhas_repetidas(unidades_brutas))
unidades = limpar_unidades(unidades_brutas)
pagina = next(u for u in unidades if u.pagina == unidades_brutas[PAGINA].pagina)
print("\nDEPOIS DA LIMPEZA:\n", pagina.texto[:800])

## Etapa 3: Chunking

Quantos chunks saem, qual o tamanho deles e como fica o cabeçalho contextual.

In [ ]:
from rag.chunking import dividir, criar_splitter

chunks = dividir(fonte, unidades, doc_hash="demo")
tamanhos = [c.metadados["n_caracteres"] for c in chunks]
print(f"{len(chunks)} chunks | min {min(tamanhos)} | média {sum(tamanhos)//len(tamanhos)} | máx {max(tamanhos)} caracteres\n")
exemplo = chunks[len(chunks) // 3]
print(exemplo.texto)
print("\nmetadados:", exemplo.metadados)

In [ ]:
# Experimente: a mesma página com chunks menores
splitter = criar_splitter(tamanho=400, sobreposicao=60)
for i, p in enumerate(splitter.split_text(pagina.texto)[:3], 1):
    print(f"--- pedaço {i} ({len(p)} car.) ---\n{p}\n")

## Etapa 4: Embeddings

Um embedding é um vetor. Frases com o mesmo sentido ficam próximas (cosseno alto), mesmo com palavras diferentes e até em outro idioma.

In [ ]:
from rag.embeddings import criar_embeddings, cosseno

emb = criar_embeddings()
frases = [
    "reserva de emergência",
    "dinheiro guardado para imprevistos",
    "emergency savings fund",
    "violência patrimonial contra a mulher",
]
vetores = emb.embed_documents(frases)
print("dimensão do vetor:", len(vetores[0]))
print("primeiros números:", [round(x, 4) for x in vetores[0][:8]], "...\n")
for i in range(1, len(frases)):
    print(f"cos('{frases[0]}', '{frases[i]}') = {cosseno(vetores[0], vetores[i]):.3f}")

In [ ]:
# Compare com o baseline lexical: ele só enxerga palavras em comum
from rag.embeddings import EmbeddingsHash
v = EmbeddingsHash().embed_documents(frases)
for i in range(1, len(frases)):
    print(f"[lexical] cos('{frases[0]}', '{frases[i]}') = {cosseno(v[0], v[i]):.3f}")

## Etapa 5: Banco vetorial

Abre a coleção persistente criada por `python ingerir.py`.

In [ ]:
from rag import banco_vetorial

db = banco_vetorial.abrir()
contagem = banco_vetorial.contagem_por_fonte(db)
print(f"Coleção: {config.COLLECTION_NAME}")
print(f"Total: {sum(contagem.values())} chunks de {len(contagem)} fontes\n")
for fonte_id, n in contagem.most_common():
    print(f"{n:5d}  {fonte_id}")

## Etapa 6: Recuperação

Veja **todos** os candidatos e quais passaram pelos filtros (limiar e máximo por fonte).

In [ ]:
from rag.recuperacao import recuperar

pergunta = "As mulheres pagam juros mais altos que os homens no crédito?"
selecionados, candidatos = recuperar(db, pergunta)
ids_sel = {t.chunk_id for t in selecionados}

print(f"Pergunta: {pergunta}\n")
for t in candidatos[:15]:
    marca = "→ LLM" if t.chunk_id in ids_sel else "     "
    print(f"{marca}  #{t.rank:2d}  sim={t.similaridade:.3f}  {t.fonte_id:40}  {t.localizador}")

In [ ]:
print(selecionados[0].referencia(), "\n")
print(selecionados[0].texto)

## Etapa 7: Geração fundamentada

O prompt que o LLM recebe e a resposta com citações.

In [ ]:
from rag.geracao import montar_contexto, PROMPT_SISTEMA, gerar

print(PROMPT_SISTEMA)
print("CONTEXTO (início):\n", montar_contexto(selecionados)[:1500], "...")

In [ ]:
resposta = gerar(pergunta, selecionados)
print(resposta.texto)

## Etapa 8: Evidências

A verificação automática das citações e o registro rastreável.

In [ ]:
from rag import evidencias

print("Fundamentada:", resposta.fundamentada)
print("Citados:", resposta.citados, "| inválidos:", resposta.citacoes_invalidas, "\n")
for n in resposta.citados:
    t = resposta.trechos[n - 1]
    print(f"[{n}] {t.referencia()}\n     {t.url}")

md_path, json_path = evidencias.salvar(resposta, candidatos)
print("\nRegistro salvo em", md_path)

## Teste o limite: pergunta fora do escopo

Um bom RAG admite quando não sabe. Observe as similaridades (mais baixas) e a resposta.

In [ ]:
p = "Qual é a melhor criptomoeda para comprar este mês?"
sel, cand = recuperar(db, p)
print("similaridade máxima:", cand[0].similaridade if cand else None)
print()
print(gerar(p, sel).texto)